# Accessing the baubook API from Python

This notebook shows a minimal workflow for accessing the baubook v3 API.

We will:

1. load credentials from a local `credentials.json` file,
2. create an authenticated API session,
3. fetch product data,
4. fetch category data,
5. fetch reference values,
6. convert selected results to `pandas` DataFrames.

In [1]:
import pandas as pd

import pexl.io.baubook  as baubook# (assuming baubook.py file in same folder)

## 1. Credentials

The API credentials are stored outside the notebook in a small JSON file.

The file should be called `credentials.json` and should be located in the same folder as this notebook.

Example structure:

```json
{
    "user": "your_username",
    "api_key": "your_password_or_api_key"
}
```
**Do not commit this file to GIT!**

In [3]:
u, pw = baubook.load_credentials("credentials.json")

print("user:", u)
print("password/api key:", pw[:4] + "..." + pw[-4:]) # never print or save secrets

user: technikum-wien-V3
password/api key: gijh...nzom


## 2. Create an authenticated session

baubook uses an authentication endpoint to create a token.

The helper function `create_baubook_session()` sends the login request and stores the token in a reusable `requests.Session`.

All following API requests use this session.

In [4]:
session = baubook.create_baubook_session(u, pw)

session

## 3. Fetch product data

The products endpoint returns a batch of baubook products.

The parameter `b_size` controls how many products are returned in one request.

In [5]:

products = baubook.get_products(session, b_size=5)

products.keys()

dict_keys(['@id', 'batching', 'items', 'items_total'])

## 4. Inspect one product

The products are stored under the key `items`.

Each item is a dictionary containing the available product metadata.

In [6]:
products["items"][0]

{'UID': '2c953cffe93b419f973343afa9da86d8',
 'bauteilaufbau': None,
 'beschreibung': {'de': 'Klebespachtel zum Kleben der Dämmplatten und zum Einbetten des Textilglasgittergewebes im webertherm WDVS'},
 'bezeichnung': {'de': 'webertherm family GROB BlueComfort'},
 'description': 'Klebespachtel zum Kleben der Dämmplatten und zum Einbetten des Textilglasgittergewebes im webertherm WDVS',
 'dicke': None,
 'einsatz_bis': None,
 'einsatz_von': None,
 'entsorgung': None,
 'fensteraufbau': None,
 'gelistet_seit_v3': '2026-03-16T07:47:21+00:00',
 'hersteller': {'Firmenname': 'Saint-Gobain Austria GmbH (weber)',
  'Firmennummer': '3737',
  'Hersteller_id': '1823760363'},
 'id': '2142747930',
 'index': None,
 'kategorie': {'grob': 'Opake Bauprodukte', 'node_id': ['8522', '8526']},
 'lambda_v3': None,
 'lin_waermebrueckenkoef': None,
 'my': {'Feucht': None,
  'Maximum': None,
  'Minimum': None,
  'Nennwert': '25',
  'Trocken': None},
 'oekokennzahlen': None,
 'produkt_index': '3737 bq',
 'rahmen'

## 5. Convert products to a table

For analysis, it is often useful to convert the JSON response to a `pandas` DataFrame.

`pandas.json_normalize()` flattens nested dictionary structures into table columns.

In [7]:
products_df = pd.json_normalize(products["items"])

products_df.head()

,UID,bauteilaufbau,description,dicke,einsatz_bis,einsatz_von,entsorgung,fensteraufbau,gelistet_seit_v3,id,...,my.Feucht,my.Maximum,my.Minimum,my.Nennwert,my.Trocken,roh_bzw_schuettdichte.Bezeichnung,roh_bzw_schuettdichte.Maximum,roh_bzw_schuettdichte.Minimum,roh_bzw_schuettdichte.Nennwert,my
0,2c953cffe93b419f973343afa9da86d8,None,Klebespachtel zum Kleben der Dämmplatten und z...,None,None,None,None,None,2026-03-16T07:47:21+00:00,2142747930,...,NaN,NaN,NaN,25,NaN,NaN,NaN,NaN,NaN,NaN
1,a15764c8900b47c890e124e608934ad3,None,Klebespachtel zum Kleben der Dämmplatten und z...,None,None,None,None,None,2026-03-16T08:14:24+00:00,2142747933,...,NaN,NaN,NaN,25,NaN,NaN,NaN,NaN,NaN,NaN
2,f5f07a7a886b45a591ca55467ade98e2,None,Klebe- und Armierungsspachtel für das weberthe...,None,None,None,None,None,2026-03-16T07:55:13+00:00,2142747935,...,NaN,NaN,NaN,25,NaN,NaN,NaN,NaN,NaN,NaN
3,bc4f07cd898d473bba903df9502722c5,None,Spezialkleber für Dämmplatten auf Holzuntergrü...,None,None,None,None,None,2026-03-16T08:10:37+00:00,2142747937,...,NaN,NaN,NaN,30,NaN,Bei Schüttungen aus kornförmigen Feststoffen i...,NaN,NaN,1600,NaN
4,86fd798da4db45ffa09d84e5284ed375,None,Komfortabel zu verarbeitende Klebespachtel zum...,None,None,None,None,None,2026-03-16T08:03:58+00:00,2142749365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Fetch a larger product batch

The API supports batch requests.

Here we request up to 150 products in one call.

In [8]:
products = baubook.get_products(session, b_size=150)

products_df = pd.json_normalize(products["items"])

products_df.head()

,UID,bauteilaufbau,description,dicke,einsatz_bis,einsatz_von,entsorgung,fensteraufbau,gelistet_seit_v3,id,...,richtwert.5.BP.Rahmen,richtwert.5.BP.Roh_bzw_Schuettdichte,richtwert.5.BOE.Bezeichnung.en,richtwert.5.BOE.Beschreibung.de,richtwert.5.BP.Dicke.Dickenbereich.max,richtwert.5.BP.Dicke.Dickenbereich.min,rahmen.Rahmen_oben_seitlich.Breite,rahmen.Rahmen_oben_seitlich.Uf,rahmen.Rahmen_unten.Breite,rahmen.Rahmen_unten.Uf
0,2c953cffe93b419f973343afa9da86d8,None,Klebespachtel zum Kleben der Dämmplatten und z...,None,None,None,None,None,2026-03-16T07:47:21+00:00,2142747930,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,a15764c8900b47c890e124e608934ad3,None,Klebespachtel zum Kleben der Dämmplatten und z...,None,None,None,None,None,2026-03-16T08:14:24+00:00,2142747933,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,f5f07a7a886b45a591ca55467ade98e2,None,Klebe- und Armierungsspachtel für das weberthe...,None,None,None,None,None,2026-03-16T07:55:13+00:00,2142747935,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bc4f07cd898d473bba903df9502722c5,None,Spezialkleber für Dämmplatten auf Holzuntergrü...,None,None,None,None,None,2026-03-16T08:10:37+00:00,2142747937,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,86fd798da4db45ffa09d84e5284ed375,None,Komfortabel zu verarbeitende Klebespachtel zum...,None,None,None,None,None,2026-03-16T08:03:58+00:00,2142749365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Check how many products are available

The response contains metadata about the total number of available products.

The key `items_total` tells us how many products exist in total.

In [9]:
products.get("items_total")

4390

## 8. Fetch product categories

baubook products are organized in categories.

Endpoint used by the helper function:

```text
GET /@bb_v3_categories
```
The result is a nested category tree.

In [13]:
categories = baubook.get_categories(session)

type(categories), len(categories)


(dict, 1)

## 9. Inspect category IDs

The top-level category IDs are the keys of the returned dictionary.

In [ ]:
list(categories.keys())[:10]

['7721']

## 10. Inspect one category

A category entry contains a title and may contain nested `sub_categories`.

The title usually contains German and English labels.

In [16]:
first_category_id = list(categories.keys())[0]
first_category = categories[first_category_id]

print("Category ID:", first_category_id)
print("DE:", first_category["Titel"].get("de"))
print("EN:", first_category["Titel"].get("en"))
print()

subcategories = first_category.get("sub_categories", {})

for sub_id, sub in subcategories.items():
    title = sub.get("Titel", {})
    print(sub_id, "|", title.get("de"), "|", title.get("en"))

Category ID: 7721
DE: Produktgruppen
EN: Product groups

46318 | Dachbegrünung | green roofs
7722 | Bauprodukte | Building products
8563 | Haustechnik | Building services


## 11. Inspect subcategories recursively

Some subcategories contain further nested subcategories.

The following helper function prints the category tree with indentation.

In [17]:
def print_category_tree(category_dict, level=0):
    for category_id, category in category_dict.items():
        title = category.get("Titel", {})
        title_de = title.get("de", "")
        title_en = title.get("en", "")

        indent = "  " * level
        print(f"{indent}{category_id} | {title_de} | {title_en}")

        subcategories = category.get("sub_categories", {})
        if subcategories:
            print_category_tree(subcategories, level=level + 1)

In [18]:
print_category_tree({
    first_category_id: first_category
})

7721 | Produktgruppen | Product groups
  46318 | Dachbegrünung | green roofs
    46319 | Vegetationstragschichten | vegetation base course
      46320 | Substrate für Extensivbegrünungen | substrates for extensive green roofs
      46321 | Substrate für Intensivbegrünungen | substrates for intensive green roofs
    46322 | Filterschichten | filter layer
    46323 | Drän- und Speicherschichten | water storage/ drainage layer
      50249 | minerlische Schüttstoffe | mineral bulk materials
      50250 | Drän- und Wasserspeicherplatten/-matten | drainage and water storage boards/mats
    46324 | Schutz - und Trennschichten | protective layer
    46325 | kombinierte Schichten für Gründächer (Filter- und Dränschicht) | combined layers for green roofs (filter and drainage layer)
    50327 | Wurzelfeste Abdichtungsbahnen | root-resistant sealing membrane
  7722 | Bauprodukte | Building products
    15250 | Wärmedämmverbundsysteme | External thermal insulation composite system
      18094 | WDV

## 12. Convert the category tree to a table

For analysis, it is often more useful to convert the nested category tree into a flat table.

Each row represents one category and contains:

- category ID,
- parent category ID,
- hierarchy level,
- German title,
- English title.

In [19]:
def flatten_categories(category_dict, parent_id=None, level=0):
    rows = []

    for category_id, category in category_dict.items():
        title = category.get("Titel", {})

        rows.append({
            "category_id": category_id,
            "parent_id": parent_id,
            "level": level,
            "title_de": title.get("de"),
            "title_en": title.get("en"),
        })

        subcategories = category.get("sub_categories", {})
        rows.extend(
            flatten_categories(
                subcategories,
                parent_id=category_id,
                level=level + 1,
            )
        )

    return rows

In [20]:
category_rows = flatten_categories(categories)

categories_df = pd.DataFrame(category_rows)

categories_df.head(20)

,category_id,parent_id,level,title_de,title_en
0,7721,None,0,Produktgruppen,Product groups
1,46318,7721,1,Dachbegrünung,green roofs
2,46319,46318,2,Vegetationstragschichten,vegetation base course
3,46320,46319,3,Substrate für Extensivbegrünungen,substrates for extensive green roofs
4,46321,46319,3,Substrate für Intensivbegrünungen,substrates for intensive green roofs
5,46322,46318,2,Filterschichten,filter layer
6,46323,46318,2,Drän- und Speicherschichten,water storage/ drainage layer
7,50249,46323,3,minerlische Schüttstoffe,mineral bulk materials
8,50250,46323,3,Drän- und Wasserspeicherplatten/-matten,drainage and water storage boards/mats
9,46324,46318,2,Schutz - und Trennschichten,protective layer


### Excel export

In [ ]:
categories_df.to_excel("baubook_categories.xlsx", index=False)

## 13. Get information for one individual product

The product overview endpoint returns a list of products, but not necessarily all detailed information for each product.

To inspect a specific product, we first take one product from the overview list and look for its identifiers and links.

Useful fields are usually:

- `UID`
- `@id`
- `title` or `name`

The exact available fields depend on the API response.

In [21]:
products = baubook.get_products(session, b_size=5)

first_product = products["items"][0]

first_product

{'UID': '2c953cffe93b419f973343afa9da86d8',
 'bauteilaufbau': None,
 'beschreibung': {'de': 'Klebespachtel zum Kleben der Dämmplatten und zum Einbetten des Textilglasgittergewebes im webertherm WDVS'},
 'bezeichnung': {'de': 'webertherm family GROB BlueComfort'},
 'description': 'Klebespachtel zum Kleben der Dämmplatten und zum Einbetten des Textilglasgittergewebes im webertherm WDVS',
 'dicke': None,
 'einsatz_bis': None,
 'einsatz_von': None,
 'entsorgung': None,
 'fensteraufbau': None,
 'gelistet_seit_v3': '2026-03-16T07:47:21+00:00',
 'hersteller': {'Firmenname': 'Saint-Gobain Austria GmbH (weber)',
  'Firmennummer': '3737',
  'Hersteller_id': '1823760363'},
 'id': '2142747930',
 'index': None,
 'kategorie': {'grob': 'Opake Bauprodukte', 'node_id': ['8522', '8526']},
 'lambda_v3': None,
 'lin_waermebrueckenkoef': None,
 'my': {'Feucht': None,
  'Maximum': None,
  'Minimum': None,
  'Nennwert': '25',
  'Trocken': None},
 'oekokennzahlen': None,
 'produkt_index': '3737 bq',
 'rahmen'

In [22]:
first_product.keys()

dict_keys(['UID', 'bauteilaufbau', 'beschreibung', 'bezeichnung', 'description', 'dicke', 'einsatz_bis', 'einsatz_von', 'entsorgung', 'fensteraufbau', 'gelistet_seit_v3', 'hersteller', 'id', 'index', 'kategorie', 'lambda_v3', 'lin_waermebrueckenkoef', 'my', 'oekokennzahlen', 'produkt_index', 'rahmen', 'richtwert', 'roh_bzw_schuettdichte', 'sd', 'spez_waermekapazitaet', 'title', 'ud'])

## 19. Reference values

Reference values are lookup tables used by baubook.

They are useful because product data often contains IDs instead of directly readable labels.

The endpoint pattern is:

```text
GET /@bb_v3_referencevalues/{reference_type}

Example:

GET /@bb_v3_referencevalues/ZG
```
This returns all reference values of type ZG.

In [26]:
zg_values = baubook.get_reference_values(session, "ZG")

zg_values.keys()

dict_keys(['items', 'type'])

## 20. Inspect the reference value items

The actual reference values are stored in the `items` field.

Each item has an ID and a dictionary of properties.

In [27]:
zg_items = zg_values["items"]

list(zg_items.keys())[:5]

['2142740773', '2142740774', '2142740776', '2142740777', '2142740779']

In [28]:
first_zg_id = list(zg_items.keys())[0]

first_zg_id, zg_items[first_zg_id]

('2142740773',
 {'BOE_Richtwert_id': '2142700410',
  'BP_Richtwert_id': '2142697131',
  'Bezeichnung': {'de': 'Aluminium (2-IV; Ug <1,4; Uf <1,4)',
   'en': 'Aluminium (2-IV; Ug <1.4; Uf <1.4)'},
  'Liste_Nutzungsdauer': {'Nutzungsdauer_Katalog_id': '2',
   'Nutzungsdauer_id': '1999'},
  'Schraffur_id': '0'})

## 21. Convert reference values to a table

The reference value response is a dictionary of dictionaries.

We convert it to a DataFrame and keep the reference IDs as a column.

In [29]:
zg_df = pd.DataFrame.from_dict(zg_items, orient="index")

zg_df = zg_df.reset_index(names="reference_id")

zg_df.head()

,reference_id,BOE_Richtwert_id,BP_Richtwert_id,Bezeichnung,Liste_Nutzungsdauer,Schraffur_id,Bezeichnung_kurz
0,2142740773,2142700410,2142697131,"{'de': 'Aluminium (2-IV; Ug <1,4; Uf <1,4)', '...","{'Nutzungsdauer_Katalog_id': '2', 'Nutzungsdau...",0,NaN
1,2142740774,2142725760,2142697131,"{'de': 'Aluminium (2-IV; Ug <1,4; Uf <1,4)', '...","{'Nutzungsdauer_Katalog_id': '2', 'Nutzungsdau...",0,NaN
2,2142740776,2142700410,2142696351,"{'de': 'Aluminium (2-IV; Ug <1,4; Uf 1,4 - 2,...","{'Nutzungsdauer_Katalog_id': '2', 'Nutzungsdau...",0,NaN
3,2142740777,2142725760,2142696351,"{'de': 'Aluminium (2-IV; Ug <1,4; Uf 1,4 - 2,...","{'Nutzungsdauer_Katalog_id': '2', 'Nutzungsdau...",0,NaN
4,2142740779,2142700410,2142696353,"{'de': 'Aluminium (2-IV; Ug <1,4; Uf >2,1)', '...","{'Nutzungsdauer_Katalog_id': '2', 'Nutzungsdau...",0,NaN


## 22. Extract readable German and English labels

Many reference values store names in nested dictionaries, for example:

```python
{"de": "...", "en": "..."}

In [30]:
if "Bezeichnung" in zg_df.columns:
    zg_df["label_de"] = zg_df["Bezeichnung"].apply(
        lambda x: x.get("de") if isinstance(x, dict) else None
    )
    zg_df["label_en"] = zg_df["Bezeichnung"].apply(
        lambda x: x.get("en") if isinstance(x, dict) else None
    )

zg_df[["reference_id", "label_de", "label_en"]].head()

,reference_id,label_de,label_en
0,2142740773,"Aluminium (2-IV; Ug <1,4; Uf <1,4)",Aluminium (2-IV; Ug <1.4; Uf <1.4)
1,2142740774,"Aluminium (2-IV; Ug <1,4; Uf <1,4)",Aluminium (2-IV; Ug <1.4; Uf <1.4)
2,2142740776,"Aluminium (2-IV; Ug <1,4; Uf 1,4 - 2,1)",Aluminium (2-IV; Ug <1.4; Uf 1.4 - 2.1)
3,2142740777,"Aluminium (2-IV; Ug <1,4; Uf 1,4 - 2,1)",Aluminium (2-IV; Ug <1.4; Uf 1.4 - 2.1)
4,2142740779,"Aluminium (2-IV; Ug <1,4; Uf >2,1)",Aluminium (2-IV; Ug <1.4; Uf >2.1)


## 26. Export all products to Excel

The product endpoint is paginated. This means the API does not return all products at once by default.

To export all products, we:

1. request the first batch,
2. read `items_total`,
3. request all batches until all products are collected,
4. convert the list to a `pandas` DataFrame,
5. save the table as an Excel file.

In [31]:
all_products = []

batch_size = 150
b_start = 0

while True:
    batch = baubook.get_products(
        session,
        b_size=batch_size,
        b_start=b_start,
    )

    items = batch.get("items", [])
    total = batch.get("items_total", len(items))

    all_products.extend(items)

    print(f"Fetched {len(all_products)} / {total}")

    if len(all_products) >= total:
        break

    b_start += batch_size

len(all_products)

Fetched 150 / 4390
Fetched 300 / 4390
Fetched 450 / 4390
Fetched 600 / 4390
Fetched 750 / 4390
Fetched 900 / 4390
Fetched 1050 / 4390
Fetched 1200 / 4390
Fetched 1350 / 4390
Fetched 1500 / 4390
Fetched 1650 / 4390
Fetched 1800 / 4390
Fetched 1950 / 4390
Fetched 2100 / 4390
Fetched 2250 / 4390
Fetched 2400 / 4390
Fetched 2550 / 4390
Fetched 2700 / 4390
Fetched 2850 / 4390
Fetched 3000 / 4390
Fetched 3150 / 4390
Fetched 3300 / 4390
Fetched 3450 / 4390
Fetched 3600 / 4390
Fetched 3750 / 4390
Fetched 3900 / 4390
Fetched 4050 / 4390
Fetched 4200 / 4390
Fetched 4350 / 4390
Fetched 4390 / 4390


4390

In [32]:
all_products_df = pd.json_normalize(all_products)

all_products_df.head()

,UID,bauteilaufbau,description,dicke,einsatz_bis,einsatz_von,entsorgung,fensteraufbau,gelistet_seit_v3,id,...,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.A1.Wert.FW,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.A2.Wert.FW,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.A2.Wert.GWP-B,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.A3.Wert.FW,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.A3.Wert.GWP-B,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.A3.Wert.PERM,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.D.Wert.EEE,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.D.Wert.EET,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.D aus C.Wert.EEE,oekokennzahlen.Welt.EI-A2.Lebensphase_Szenario.D aus C.Wert.EET
0,2c953cffe93b419f973343afa9da86d8,NaN,Klebespachtel zum Kleben der Dämmplatten und z...,None,None,None,NaN,None,2026-03-16T07:47:21+00:00,2142747930,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,a15764c8900b47c890e124e608934ad3,NaN,Klebespachtel zum Kleben der Dämmplatten und z...,None,None,None,NaN,None,2026-03-16T08:14:24+00:00,2142747933,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,f5f07a7a886b45a591ca55467ade98e2,NaN,Klebe- und Armierungsspachtel für das weberthe...,None,None,None,NaN,None,2026-03-16T07:55:13+00:00,2142747935,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bc4f07cd898d473bba903df9502722c5,NaN,Spezialkleber für Dämmplatten auf Holzuntergrü...,None,None,None,NaN,None,2026-03-16T08:10:37+00:00,2142747937,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,86fd798da4db45ffa09d84e5284ed375,NaN,Komfortabel zu verarbeitende Klebespachtel zum...,None,None,None,NaN,None,2026-03-16T08:03:58+00:00,2142749365,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
all_products_df.to_excel("baubook_all_products.xlsx", index=False)